# パックマン：報酬・DQN/SARSA・PSOの比較
1番で全テストとDriveへのコード保存、2番で全比較を開始・再開。
横軸は学習回数（エピソード数）。250条件×5seedとPSO等の探索・独立再学習。
計画はall_plan.json、画像はresults/factors/figures/とresults/search/以下。
Colab切断後は再接続して1番→2番。同一版の保存済み実験を復元します。


In [ ]:
#@title 1. 比較コードを展開・全テスト・Driveへ保存
import base64, hashlib, importlib.util, io, json, os, platform, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version
EXPECTED_BUNDLE = "9e75afec6a09a84cf5199b2a1bc1d612747d9ad7369f7a50907222189759b0fd"
WORK = Path("/content/pacman_comparison_gpu_code")
LOCAL = Path("/content/pacman_comparison_gpu_study")
FOLDER = Path("/content/drive/MyDrive/DQN_Pacman_comparisons_gpu_20261020_9e75afec")
archive = base64.b64decode("...")
assert hashlib.sha256(archive).hexdigest() == EXPECTED_BUNDLE
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    frozen = json.loads(z.read("source_hashes.json"))
    assert set(z.namelist()) == set(frozen) | {"source_hashes.json"}
    for name, expected in frozen.items():
        assert not Path(name).is_absolute() and ".." not in Path(name).parts
        data = z.read(name)
        assert hashlib.sha256(data).hexdigest() == expected, name
        target = WORK / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
print("SOURCE_VERIFIED:", len(frozen), "files; bundle", EXPECTED_BUNDLE, flush=True)
missing = [p for p in ("numpy", "gymnasium", "torch", "pytest", "matplotlib", "PIL")
           if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *["Pillow" if p == "PIL" else p for p in missing]], check=True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PACMAN_DEVICE"] = "cpu"
os.environ["PACMAN_TORCH_THREADS"] = "1"
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before deployment"
gpu = torch.cuda.get_device_properties(0)
hardware = dict(gpu=gpu.name, gpu_memory_gib=gpu.total_memory/1024**3,
                cpu_count=os.cpu_count(), cuda=torch.version.cuda,
                ram_gib=os.sysconf("SC_PAGE_SIZE")*os.sysconf("SC_PHYS_PAGES")/1024**3)
print("GPU_HARDWARE:", json.dumps(hardware), flush=True)
test = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                      cwd=WORK, capture_output=True, text=True)
test_log = test.stdout + "\n" + test.stderr
(WORK / "deployment_tests.log").write_text(test_log, encoding="utf-8")
print(test_log, flush=True)
test.check_returncode()
os.environ["PACMAN_DEVICE"] = "cuda"
subprocess.run([sys.executable, "-u", "comparison_study.py", "all", "--out", str(LOCAL), "--plan-only"],
               cwd=WORK, check=True)
from google.colab import drive
drive.mount("/content/drive")
manifest_path = FOLDER / "source_hashes.json"
if FOLDER.exists() and any(FOLDER.iterdir()) and not manifest_path.exists():
    raise RuntimeError("Destination exists without a manifest; use a new folder")
if manifest_path.exists() and json.loads(manifest_path.read_text()) != frozen:
    raise RuntimeError("Existing source differs; use a new deployment folder")
FOLDER.mkdir(parents=True, exist_ok=True)
for name, expected in frozen.items():
    target = FOLDER / "source" / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(WORK / name, target)
    assert hashlib.sha256(target.read_bytes()).hexdigest() == expected, name
manifest_path.write_text(json.dumps(frozen, indent=1), encoding="utf-8")
meta = dict(bundle_sha256=EXPECTED_BUNDLE, tests_passed=True, source_files=len(frozen),
            deployed_at=datetime.now(timezone.utc).isoformat(), python=platform.python_version(),
            packages={p:version(p) for p in ("torch", "numpy", "gymnasium", "pytest", "matplotlib")},
            local=str(LOCAL), drive=str(FOLDER))
(FOLDER / "deployment.json").write_text(json.dumps(meta, indent=1), encoding="utf-8")
shutil.copy2(WORK / "deployment_tests.log", FOLDER / "deployment_tests.log")
shutil.copy2(LOCAL / "all_plan.json", FOLDER / "all_plan.json")
(FOLDER / "hardware.json").write_text(json.dumps(hardware, indent=1))
print("GPU_DEPLOY_READY:", FOLDER, hardware, flush=True)
print("66比較項目・250条件×5seed。全画像の横軸は学習回数。次のセルで全比較を開始・再開します。", flush=True)


In [ ]:
#@title 2. 全比較を開始・再開（完了まで実行中）
START_TRAINING = True #@param {type:"boolean"}
WORKERS = 8 #@param {type:"integer"}
REPORT_EVERY = 1 #@param {type:"integer"}
if START_TRAINING:
    import hashlib, json, os, subprocess, sys
    from pathlib import Path
    WORK = Path("/content/pacman_comparison_gpu_code")
    LOCAL = Path("/content/pacman_comparison_gpu_study")
    FOLDER = Path("/content/drive/MyDrive/DQN_Pacman_comparisons_gpu_20261020_9e75afec")
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["PACMAN_DEVICE"] = "cuda"
    os.environ["PACMAN_TORCH_THREADS"] = "1"
    import torch
    assert torch.cuda.is_available(), "GPU runtime required; CPU fallback is disabled"
    WORKERS = min(WORKERS, max(1, (os.cpu_count() or 2)-2))
    print("CUDA_TRAINING:", torch.cuda.get_device_name(0), "workers", WORKERS, flush=True)
    meta = json.loads((FOLDER / "deployment.json").read_text())
    assert meta["tests_passed"] and meta["bundle_sha256"] == "9e75afec6a09a84cf5199b2a1bc1d612747d9ad7369f7a50907222189759b0fd", "Run cell 1 first"
    frozen = json.loads((FOLDER / "source_hashes.json").read_text())
    for name, expected in frozen.items():
        assert hashlib.sha256((WORK / name).read_bytes()).hexdigest() == expected, name
        assert hashlib.sha256((FOLDER / "source" / name).read_bytes()).hexdigest() == expected, name
    available = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_AVPHYS_PAGES") / 1024**3
    if not 1 <= WORKERS <= 8 or available < WORKERS * 3 + 4:
        raise RuntimeError(f"workers={WORKERS}, available={available:.1f}GiB: reduce parallelism")
    assert REPORT_EVERY >= 1
    command = [sys.executable, "-u", "comparison_study.py", "all", "--out", str(LOCAL),
               "--backup", str(FOLDER / "results"), "--workers", str(WORKERS),
               "--report-every", str(REPORT_EVERY)]
    print("全因子比較 → 報酬/γ/PSO/探索法比較。切断後は同じセルで再開。", flush=True)
    print("保存先:", FOLDER / "results", flush=True)
    with (WORK / "full_study.log").open("a", buffering=1) as log:
        process = subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        print("COMPARISON_PID:", process.pid, flush=True)
        for line in process.stdout:
            log.write(line)
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f"exit={code}: inspect /content/pacman_comparison_gpu_code/full_study.log")
    print("ALL_COMPARISONS_FINISHED: 各COMPLETE.jsonのfailed_trialsも確認してください。", flush=True)
else:
    print("未開始。START_TRAININGにチェックして実行すると開始・再開します。")
